# TTAExp — NIDS continual test-time adaptation (CTTA)

Thin notebook: every shared definition now lives in the [`nids-toolkit`](https://github.com/braim/nids-toolkit) package. This notebook is only **configuration** + the **main execution loop**.

`TTAExp` and `TTAExp50` are identical except the config cell (`few_shot_ratio`, `min_pool_size`).

In [ ]:
# Install the shared toolkit (pulls efficient-kan + the scientific stack).
!pip install -q git+https://github.com/braim/nids-toolkit.git

## Configuration

In [ ]:
from nids_toolkit import ExperimentConfig, run_grid, run_sequential, DEFAULT_DATASETS

# The ONLY thing that differs between TTAExp and TTAExp50 is this config block.
cfg = ExperimentConfig(
    # data / model
    sample_n=1_000_000,
    latent_dim=32,
    # loss / scaling
    loss_type="weighted_ce",
    scaler_type="quantile",
    # few-shot pool + CTTA
    few_shot_ratio=1e-4,     # TTAExp
    min_pool_size=100,          # TTAExp
    # AGSA (Activation-Gated Spline Adaptation) — the KAN-only novelty
    use_spline_gate=True,
    spline_true=True,
)
cfg

## Run the episodic grid

Writes `grid_results.csv` and `ablation_results.csv`.

In [ ]:
results_df, ablation_df = run_grid(
    cfg,
    architectures=["kan", "cnn", "tab", "flowtransformer"],
    datasets=DEFAULT_DATASETS,
)
results_df

## (optional) Sequential / continual protocol

Adapt through a domain *sequence* with no reset between targets (source → target1 → target2).

In [ ]:
seq_df = run_sequential(cfg)   # continual protocol; writes sequential_results.csv
seq_df